# Offline Reward-Head Fitting (Env Task Reward)

This notebook collects offline rollout data from a checkpointed SSL policy, evaluates current reward-head prediction quality, then tunes **only** the reward prediction head.

Workflow:
1. Load checkpoint + run config
2. Collect ~30,000 timesteps: `(obs_t, action_t, env_reward_t)`
3. Train/eval split and baseline metrics
4. Reward-head-only fine-tuning with per-epoch logs
5. Save tuned head and metrics artifacts

In [ ]:
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
import gymnasium as gym
import matplotlib.pyplot as plt

from airhockey import AirHockeyEnv
from scripts.smooth_policy.agent import Agent
from scripts.smooth_policy.amp_history.amp_training.ssl_modules import (
    SharedStateEncoder,
    ActionConditionedRewardHead,
)

# ------------------------
# User-configurable values
# ------------------------
CHECKPOINT_DIR = Path('/home/air-hockey/daliu/air-hockey-rl/runs/ssl/test/with_discriminatorr4/checkpoint_280')
RUN_DIR = CHECKPOINT_DIR.parent
TARGET_TIMESTEPS = 30_000
TRAIN_RATIO = 0.8
BATCH_SIZE = 1024
EPOCHS = 120
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
SEED = 7
FORCE_DEVICE = None  # e.g. 'cuda:0' or 'cpu'; None = auto
PRINT_EVERY = 5

OUT_DIR = CHECKPOINT_DIR / 'reward_head_offline_fit'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Checkpoint:', CHECKPOINT_DIR)
print('Output dir :', OUT_DIR)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pick_device(force_device=None):
    if force_device is not None:
        return torch.device(force_device)
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def build_policy_observation_from_latent(latent, last_action, use_last_action):
    if not use_last_action:
        return latent
    return torch.cat([latent, last_action], dim=-1)


def make_env_fn(air_hockey_config: dict, env_seed: int):
    def _thunk():
        cfg = copy.deepcopy(air_hockey_config)
        cfg['seed'] = int(env_seed)
        return AirHockeyEnv(cfg)
    return _thunk


def r2_score_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 1e-12:
        return float('nan')
    return float(1.0 - ss_res / ss_tot)


def pearson_corr_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)
    y_true_std = np.std(y_true)
    y_pred_std = np.std(y_pred)
    if y_true_std <= 1e-12 or y_pred_std <= 1e-12:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    return {
        'mse': mse,
        'mae': mae,
        'r2': r2_score_np(y_true, y_pred),
        'pearson': pearson_corr_np(y_true, y_pred),
    }


set_seed(SEED)
DEVICE = pick_device(FORCE_DEVICE)
print('Using device:', DEVICE)

In [ ]:
# Load run args + env config used for the checkpoint
args_path = RUN_DIR / 'args.yaml'
config_path = RUN_DIR / 'config.yaml'

with open(args_path, 'r') as f:
    run_args = yaml.safe_load(f)
with open(config_path, 'r') as f:
    run_config = yaml.safe_load(f)

air_hockey_cfg = run_config['air_hockey']
num_envs = int(run_args['num_envs'])
obs_dim = 30 if air_hockey_cfg.get('obs_type', 'history') == 'history' else None

# Build vectorized envs
env_fns = [
    make_env_fn(air_hockey_cfg, env_seed=SEED + 17 * i)
    for i in range(num_envs)
]
envs = gym.vector.SyncVectorEnv(env_fns)

# Infer dimensions directly from env spaces
obs_shape = envs.single_observation_space.shape
action_shape = envs.single_action_space.shape
obs_dim = int(np.prod(obs_shape))
action_dim = int(np.prod(action_shape))

policy_obs_dim = int(run_args['ssl_latent_dim']) + (
    action_dim if bool(run_args.get('use_last_action_in_policy_state', False)) else 0
)

policy_env_view = type('PolicyView', (), {
    'single_observation_space': gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(policy_obs_dim,),
        dtype=np.float32,
    ),
    'single_action_space': envs.single_action_space,
})

agent = Agent(
    policy_env_view,
    action_scale=float(run_args.get('action_scale', 1.0)),
    action_bias=0.0,
    hidden_layer_size=int(run_args['agent_hidden_layer_size']),
    num_hidden_layers=int(run_args['agent_num_hidden_layers']),
).to(DEVICE)

state_encoder = SharedStateEncoder(
    obs_dim=obs_dim,
    latent_dim=int(run_args['ssl_latent_dim']),
    hidden_layer_size=int(run_args['ssl_encoder_hidden_layer_size']),
    num_hidden_layers=int(run_args['ssl_encoder_num_hidden_layers']),
).to(DEVICE)

reward_head = ActionConditionedRewardHead(
    latent_dim=int(run_args['ssl_latent_dim']),
    action_dim=action_dim,
    hidden_layer_size=int(run_args['ssl_reward_head_hidden_layer_size']),
    num_hidden_layers=int(run_args['ssl_reward_head_num_hidden_layers']),
).to(DEVICE)

agent.load_state_dict(torch.load(CHECKPOINT_DIR / 'model.pth', map_location=DEVICE))
state_encoder.load_state_dict(torch.load(CHECKPOINT_DIR / 'state_encoder.pth', map_location=DEVICE))
reward_head.load_state_dict(torch.load(CHECKPOINT_DIR / 'reward_head.pth', map_location=DEVICE))

agent.eval()
state_encoder.eval()
reward_head.eval()

print('obs_dim:', obs_dim)
print('action_dim:', action_dim)
print('num_envs:', num_envs)
print('use_last_action_in_policy_state:', bool(run_args.get('use_last_action_in_policy_state', False)))

In [ ]:
# Collect rollout data: (obs_t, action_t, env_task_reward_t)
obs_np, _ = envs.reset(seed=SEED)
obs_t = torch.as_tensor(obs_np, dtype=torch.float32, device=DEVICE)
last_action = torch.zeros((num_envs, action_dim), dtype=torch.float32, device=DEVICE)

obs_chunks = []
action_chunks = []
reward_chunks = []

use_last_action = bool(run_args.get('use_last_action_in_policy_state', False))

collected = 0
step_count = 0

with torch.no_grad():
    while collected < TARGET_TIMESTEPS:
        latent = state_encoder(obs_t)
        policy_obs = build_policy_observation_from_latent(latent, last_action, use_last_action)
        action, _, _, _ = agent.get_action_and_value(policy_obs)

        next_obs_np, reward_np, terminations, truncations, _ = envs.step(action.cpu().numpy())
        done = np.logical_or(terminations, truncations)

        obs_chunks.append(obs_t.detach().cpu())
        action_chunks.append(action.detach().cpu())
        reward_chunks.append(torch.as_tensor(reward_np, dtype=torch.float32).view(-1, 1))

        collected += num_envs
        step_count += 1

        next_obs_t = torch.as_tensor(next_obs_np, dtype=torch.float32, device=DEVICE)
        last_action = action.detach().clone()
        if done.any():
            done_mask = torch.as_tensor(done, dtype=torch.bool, device=DEVICE)
            last_action[done_mask] = 0.0

        obs_t = next_obs_t

        if step_count % 200 == 0:
            print(f'Steps: {step_count}, transitions: {collected}')

obs_all = torch.cat(obs_chunks, dim=0)[:TARGET_TIMESTEPS]
action_all = torch.cat(action_chunks, dim=0)[:TARGET_TIMESTEPS]
reward_all = torch.cat(reward_chunks, dim=0)[:TARGET_TIMESTEPS].squeeze(-1)

print('Collected transitions:', obs_all.shape[0])
print('obs shape:', tuple(obs_all.shape))
print('action shape:', tuple(action_all.shape))
print('reward shape:', tuple(reward_all.shape))

In [ ]:
# Save collected dataset + quick sanity stats
dataset_path = OUT_DIR / 'offline_reward_dataset.pt'
stats_path = OUT_DIR / 'dataset_stats.json'

dataset = {
    'obs': obs_all,
    'actions': action_all,
    'rewards': reward_all,
    'target_name': 'env_task_reward',
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'seed': int(SEED),
}
torch.save(dataset, dataset_path)

stats = {
    'num_samples': int(obs_all.shape[0]),
    'reward_mean': float(reward_all.mean().item()),
    'reward_std': float(reward_all.std(unbiased=False).item()),
    'reward_min': float(reward_all.min().item()),
    'reward_max': float(reward_all.max().item()),
    'action_abs_mean': float(action_all.abs().mean().item()),
    'action_l2_mean': float(torch.linalg.norm(action_all, dim=-1).mean().item()),
}
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)

print('Saved dataset:', dataset_path)
print('Saved stats  :', stats_path)
print(json.dumps(stats, indent=2))

In [ ]:
# Train/eval split
num_samples = obs_all.shape[0]
perm = torch.randperm(num_samples)
train_size = int(TRAIN_RATIO * num_samples)
train_idx = perm[:train_size]
eval_idx = perm[train_size:]

train_obs = obs_all[train_idx]
train_actions = action_all[train_idx]
train_rewards = reward_all[train_idx]

eval_obs = obs_all[eval_idx]
eval_actions = action_all[eval_idx]
eval_rewards = reward_all[eval_idx]

print(f'Train samples: {len(train_idx)}')
print(f'Eval samples : {len(eval_idx)}')


def predict_rewards(encoder, head, obs_tensor, action_tensor, batch_size=2048, device=DEVICE):
    preds = []
    encoder.eval()
    head.eval()
    with torch.no_grad():
        for start in range(0, obs_tensor.shape[0], batch_size):
            end = start + batch_size
            b_obs = obs_tensor[start:end].to(device)
            b_act = action_tensor[start:end].to(device)
            b_lat = encoder(b_obs)
            b_pred = head(b_lat, b_act)
            preds.append(b_pred.detach().cpu())
    return torch.cat(preds, dim=0).view(-1).numpy()


baseline_train_pred = predict_rewards(state_encoder, reward_head, train_obs, train_actions)
baseline_eval_pred = predict_rewards(state_encoder, reward_head, eval_obs, eval_actions)

baseline_train_metrics = compute_metrics(train_rewards.numpy(), baseline_train_pred)
baseline_eval_metrics = compute_metrics(eval_rewards.numpy(), baseline_eval_pred)

baseline_df = pd.DataFrame([
    {'split': 'train', **baseline_train_metrics},
    {'split': 'eval', **baseline_eval_metrics},
])
baseline_df

In [ ]:
# Baseline prediction visualization on eval split
plt.figure(figsize=(6, 6))
plt.scatter(eval_rewards.numpy(), baseline_eval_pred, s=8, alpha=0.25)
min_v = float(min(eval_rewards.min().item(), baseline_eval_pred.min()))
max_v = float(max(eval_rewards.max().item(), baseline_eval_pred.max()))
plt.plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1)
plt.xlabel('True env task reward')
plt.ylabel('Predicted reward')
plt.title('Baseline Reward Head: Eval Pred vs True')
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# Reward-head-only fine-tuning
for p in state_encoder.parameters():
    p.requires_grad = False
state_encoder.eval()

for p in reward_head.parameters():
    p.requires_grad = True
reward_head.train()

# Keep a snapshot to verify the encoder did not change
encoder_snapshot = {k: v.detach().cpu().clone() for k, v in state_encoder.state_dict().items()}
initial_reward_head_state = {k: v.detach().cpu().clone() for k, v in reward_head.state_dict().items()}

optimizer = torch.optim.Adam(
    reward_head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-6,
)

history = []
best_eval_mse = float('inf')
best_state = None

train_n = train_obs.shape[0]

for epoch in range(1, EPOCHS + 1):
    reward_head.train()
    perm = torch.randperm(train_n)
    epoch_losses = []

    for start in range(0, train_n, BATCH_SIZE):
        end = start + BATCH_SIZE
        idx = perm[start:end]

        b_obs = train_obs[idx].to(DEVICE)
        b_act = train_actions[idx].to(DEVICE)
        b_rew = train_rewards[idx].to(DEVICE)

        with torch.no_grad():
            b_lat = state_encoder(b_obs)

        pred = reward_head(b_lat, b_act).view(-1)
        loss = F.mse_loss(pred, b_rew)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(float(loss.item()))

    train_pred = predict_rewards(state_encoder, reward_head, train_obs, train_actions)
    eval_pred = predict_rewards(state_encoder, reward_head, eval_obs, eval_actions)

    train_metrics = compute_metrics(train_rewards.numpy(), train_pred)
    eval_metrics = compute_metrics(eval_rewards.numpy(), eval_pred)

    row = {
        'epoch': epoch,
        'train_loss_batch_mse': float(np.mean(epoch_losses)) if epoch_losses else float('nan'),
        'train_mse': train_metrics['mse'],
        'train_mae': train_metrics['mae'],
        'train_r2': train_metrics['r2'],
        'train_pearson': train_metrics['pearson'],
        'eval_mse': eval_metrics['mse'],
        'eval_mae': eval_metrics['mae'],
        'eval_r2': eval_metrics['r2'],
        'eval_pearson': eval_metrics['pearson'],
    }
    history.append(row)

    if eval_metrics['mse'] < best_eval_mse:
        best_eval_mse = eval_metrics['mse']
        best_state = {k: v.detach().cpu().clone() for k, v in reward_head.state_dict().items()}

    if epoch % PRINT_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:4d} | train_mse={train_metrics['mse']:.6f} | "
            f"eval_mse={eval_metrics['mse']:.6f} | eval_mae={eval_metrics['mae']:.6f} | "
            f"eval_r={eval_metrics['pearson']:.4f}"
        )

history_df = pd.DataFrame(history)
history_df.tail(10)

In [ ]:
# Training curves (reward prediction error logging)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history_df['epoch'], history_df['train_mse'], label='train_mse')
axes[0].plot(history_df['epoch'], history_df['eval_mse'], label='eval_mse')
axes[0].set_title('MSE')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(history_df['epoch'], history_df['train_mae'], label='train_mae')
axes[1].plot(history_df['epoch'], history_df['eval_mae'], label='eval_mae')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.2)

axes[2].plot(history_df['epoch'], history_df['train_pearson'], label='train_pearson')
axes[2].plot(history_df['epoch'], history_df['eval_pearson'], label='eval_pearson')
axes[2].set_title('Pearson Correlation')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation + save artifacts
if best_state is None:
    raise RuntimeError('No best state captured during training.')

# Evaluate "last"
last_train_pred = predict_rewards(state_encoder, reward_head, train_obs, train_actions)
last_eval_pred = predict_rewards(state_encoder, reward_head, eval_obs, eval_actions)
last_train_metrics = compute_metrics(train_rewards.numpy(), last_train_pred)
last_eval_metrics = compute_metrics(eval_rewards.numpy(), last_eval_pred)

# Evaluate "best" (by eval MSE)
reward_head_best = ActionConditionedRewardHead(
    latent_dim=int(run_args['ssl_latent_dim']),
    action_dim=action_dim,
    hidden_layer_size=int(run_args['ssl_reward_head_hidden_layer_size']),
    num_hidden_layers=int(run_args['ssl_reward_head_num_hidden_layers']),
).to(DEVICE)
reward_head_best.load_state_dict(best_state)
reward_head_best.eval()

best_train_pred = predict_rewards(state_encoder, reward_head_best, train_obs, train_actions)
best_eval_pred = predict_rewards(state_encoder, reward_head_best, eval_obs, eval_actions)
best_train_metrics = compute_metrics(train_rewards.numpy(), best_train_pred)
best_eval_metrics = compute_metrics(eval_rewards.numpy(), best_eval_pred)

# Verify encoder unchanged
max_encoder_diff = 0.0
for k, v in state_encoder.state_dict().items():
    diff = (v.detach().cpu() - encoder_snapshot[k]).abs().max().item()
    max_encoder_diff = max(max_encoder_diff, float(diff))

summary = {
    'target_name': 'env_task_reward',
    'num_samples': int(num_samples),
    'train_samples': int(train_obs.shape[0]),
    'eval_samples': int(eval_obs.shape[0]),
    'hyperparams': {
        'epochs': int(EPOCHS),
        'batch_size': int(BATCH_SIZE),
        'learning_rate': float(LEARNING_RATE),
        'weight_decay': float(WEIGHT_DECAY),
        'train_ratio': float(TRAIN_RATIO),
        'seed': int(SEED),
    },
    'baseline': {
        'train': baseline_train_metrics,
        'eval': baseline_eval_metrics,
    },
    'last': {
        'train': last_train_metrics,
        'eval': last_eval_metrics,
    },
    'best': {
        'train': best_train_metrics,
        'eval': best_eval_metrics,
    },
    'best_eval_mse': float(best_eval_mse),
    'max_encoder_param_abs_diff': float(max_encoder_diff),
}

history_csv_path = OUT_DIR / 'reward_head_tuning_history.csv'
summary_json_path = OUT_DIR / 'reward_head_offline_summary.json'
summary_yaml_path = OUT_DIR / 'reward_head_offline_summary.yaml'
best_head_path = OUT_DIR / 'reward_head_tuned_offline_best.pth'
last_head_path = OUT_DIR / 'reward_head_tuned_offline_last.pth'
initial_head_path = OUT_DIR / 'reward_head_initial_snapshot.pth'

torch.save(best_state, best_head_path)
torch.save(reward_head.state_dict(), last_head_path)
torch.save(initial_reward_head_state, initial_head_path)
history_df.to_csv(history_csv_path, index=False)
with open(summary_json_path, 'w') as f:
    json.dump(summary, f, indent=2)
with open(summary_yaml_path, 'w') as f:
    yaml.safe_dump(summary, f, sort_keys=False)

comparison_df = pd.DataFrame([
    {'stage': 'baseline_eval', **baseline_eval_metrics},
    {'stage': 'last_eval', **last_eval_metrics},
    {'stage': 'best_eval', **best_eval_metrics},
])

print('Saved:')
print(' -', best_head_path)
print(' -', last_head_path)
print(' -', initial_head_path)
print(' -', history_csv_path)
print(' -', summary_json_path)
print(' -', summary_yaml_path)
print('\nEncoder max |delta| after tuning:', max_encoder_diff)
comparison_df

In [ ]:
# Optional cleanup
envs.close()